# ⚡ LazyPredict — Step-by-Step Guide

**What is LazyPredict?**  
LazyPredict runs 40+ different ML models on your data in one go — no manual setup per model. It gives you a ranked comparison table so you can immediately see which model family performs best, before spending time tuning any single one.

**Our baseline to beat:** Logistic Regression ROC-AUC = 0.8575

---

## Step 1 — Install Dependencies

The README specifies installing with `[boost]` to include XGBoost, LightGBM and CatBoost — this also fixes the XGBoostError from the plain `pip install lazypredict`.

Run this cell once, then **restart the kernel** before continuing.

In [ ]:
# Install LazyPredict with boosting libraries (XGBoost, LightGBM, CatBoost)
# [boost] extra is required — plain install misses XGBoost and causes import errors
!pip install "lazypredict[boost]" --quiet

# Pin remaining required dependencies to the versions in requirements
!pip install \
    "scikit-learn>=1.0" \
    "pandas>=1.3" \
    "numpy>=1.21" \
    "xgboost>=1.5" \
    "lightgbm>=3.0" \
    "statsmodels>=0.13" \
    "tqdm>=4.0" \
    "joblib>=1.0" \
    --quiet

print('✅ Installation done')
print('→ Restart the kernel now, then run from Step 2 onwards')

---
## Step 2 — Verify the Install

Run this after restarting the kernel. Confirms every required package loads correctly.

In [ ]:
import importlib

required = [
    'sklearn', 'pandas', 'numpy', 'xgboost',
    'lightgbm', 'statsmodels', 'tqdm', 'joblib', 'lazypredict'
]

all_ok = True
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'unknown')
        print(f'  ✅ {pkg:<15} {version}')
    except ImportError as e:
        print(f'  ❌ {pkg:<15} NOT FOUND — {e}')
        all_ok = False

print()
print('All packages ready — proceed.' if all_ok else 'Fix missing packages before continuing.')

---
## Step 3 — Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier

print('✅ Imports done')

---
## Step 4 — Load & Split the Data

Same 80/20 stratified split as in the previous notebooks.

In [ ]:
df = pd.read_csv('features_encoded_train.csv')

X = df.drop(columns=['target_bank_account'])
y = df['target_bank_account']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training rows  : {X_train.shape[0]:,}')
print(f'Validation rows: {X_val.shape[0]:,}')
print(f'Features       : {X_train.shape[1]}')

---
## Step 5 — Run LazyPredict

One block — 40+ models trained and evaluated automatically.  
Expect **2–4 minutes** to complete.

Parameters (from the README):
- `verbose=0` — silent; change to `1` to see a progress bar per model
- `ignore_warnings=True` — suppress sklearn version warnings
- `predictions=True` — also return the raw predictions from every model

> **Note on class imbalance:** LazyPredict does not support `class_weight='balanced'`. Focus on **ROC AUC** and **Balanced Accuracy** — both are robust to the 86/14 class split. Ignore raw Accuracy.

In [ ]:
clf = LazyClassifier(
    verbose=0,
    ignore_warnings=True,
    predictions=True
)

# Pass: X_train, X_val, y_train, y_val  (in that order)
models, predictions = clf.fit(X_train, X_val, y_train, y_val)

print(f'✅ Done — {len(models)} models trained and evaluated')

---
## Step 6 — View the Results Table

Columns explained:
- **Accuracy** — % correct overall (ignore — misleading with class imbalance)
- **Balanced Accuracy** — accuracy corrected for class imbalance ✅
- **ROC AUC** — separates both classes (0.5 = random, 1.0 = perfect) ✅
- **F1 Score** — balance between catching positives and avoiding false alarms ✅
- **Time Taken** — seconds to train

In [ ]:
models_sorted = models.sort_values('ROC AUC', ascending=False)

print(f'  {"Model":<40} {"ROC AUC":>8} {"Bal. Acc":>9} {"F1":>6} {"Time(s)":>8}')
print('  ' + '-' * 76)
for name, row in models_sorted.iterrows():
    roc = row.get('ROC AUC', float('nan'))
    bal = row.get('Balanced Accuracy', float('nan'))
    f1  = row.get('F1 Score', float('nan'))
    t   = row.get('Time Taken', float('nan'))
    tag = ' ← our baseline' if str(name) == 'LogisticRegression' else ''
    try:
        print(f'  {str(name):<40} {roc:>8.4f} {bal:>9.4f} {f1:>6.4f} {t:>8.2f}{tag}')
    except Exception:
        print(f'  {str(name):<40} {"N/A":>8} {"N/A":>9} {"N/A":>6} {"N/A":>8}')

---
## Step 7 — Visualise: Top Models by ROC AUC

In [ ]:
plot_df = models_sorted.dropna(subset=['ROC AUC']).head(15)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#185FA5' if str(n) == 'LogisticRegression' else '#3B6D11'
          for n in plot_df.index]

bars = ax.barh(
    plot_df.index[::-1],
    plot_df['ROC AUC'][::-1],
    color=colors[::-1],
    edgecolor='white',
    height=0.6
)

ax.axvline(0.8575, color='#185FA5', linestyle='--', linewidth=1.2,
           label='Logistic regression baseline (0.8575)')

for bar, val in zip(bars, plot_df['ROC AUC'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8, color='#5F5E5A')

ax.set_title('Top 15 models by ROC AUC — LazyPredict', fontweight='bold', pad=12)
ax.set_xlabel('ROC AUC (higher is better)')
ax.set_xlim(plot_df['ROC AUC'].min() - 0.05, 1.02)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 8 — Summary & What to Do Next

In [ ]:
best_name = models_sorted['ROC AUC'].idxmax()
best_auc  = models_sorted['ROC AUC'].max()
baseline  = 0.8575
delta     = best_auc - baseline

print('=' * 58)
print('  LAZYPREDICT SUMMARY')
print('=' * 58)
print(f'  Models tested         : {len(models_sorted)}')
print(f'  Best model            : {best_name}')
print(f'  Best ROC AUC          : {best_auc:.4f}')
print(f'  Baseline (LogReg)     : {baseline:.4f}')
print(f'  Improvement           : {"+" if delta >= 0 else ""}{delta:.4f}')
print('=' * 58)
print()
print('Recommended next steps:')
print('  1. Pick the top 2-3 models from the table above')
print('  2. Re-train them manually with class_weight="balanced"')
print('  3. Tune hyperparameters with GridSearchCV or RandomizedSearchCV')
print()
print('  LazyPredict scores are a shortlist guide, not final scores.')
print('  Proper class balancing and tuning will change the results.')